# Phase 0 — validating the stats configuration layer

PLAN.md section 10 (Phase 0): *"Prototype the stats, not the app. Validate the effects system
(logit-space composition) and distribution parameterization (method-of-moments conversions) in a
notebook before building any UI around them."*

This notebook exercises the two modules that came out of that validation pass —
`funnel_sim.simulation.distributions` and `funnel_sim.simulation.effects` — and checks the two
things PLAN.md flags as the real risk if this layer is wrong:

1. **Distribution parameterization** — does mean/variance -> native params -> `scipy.stats`
   distribution actually recover the mean/variance the user asked for, for every family, and does
   it fail loudly (not silently) outside each family's valid domain?
2. **Effects composability** — does stacking several logit-space effects behave sanely (stay in
   `(0, 1)`, no NaN/Inf) even when they're pushed hard, and — the actual point of the whole
   app (PLAN.md section 1) — if we simulate outcomes from a *known* configuration and fit a
   logistic regression to them, does the model recover the true configured effects?

Once this holds up, the module code is what actually ships into `simulation/` — this notebook is
the scratch/validation pass, not the source of truth (PLAN.md section 13).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from funnel_sim.simulation.distributions import Family, moments_to_params, preview_xy
from funnel_sim.simulation.effects import Effect, EffectSet, apply_effects

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Distribution parameterization

One config per family we support (PLAN.md section 4), each specified the way a user would — mean and variance — never in native parameters.

In [ ]:
CONFIGS = {
    Family.NORMAL: dict(mean=12.0, variance=9.0),           # e.g. deal size around $12k, +/-$3k
    Family.GAMMA: dict(mean=3.0, variance=2.0),              # e.g. days-to-first-touch
    Family.BETA: dict(mean=0.22, variance=0.01),             # e.g. ICP-fit score in [0, 1]
    Family.LOGNORMAL: dict(mean=8000.0, variance=9_000_000.0),  # e.g. deal size, heavy right tail
    Family.POISSON: dict(mean=4.0, variance=4.0),             # e.g. tasks/day for an average rep
    Family.NEGATIVE_BINOMIAL: dict(mean=4.0, variance=9.0),   # e.g. tasks/day, over-dispersed
}

results = {family: moments_to_params(family, **cfg) for family, cfg in CONFIGS.items()}
for family, result in results.items():
    print(f"{family.value:20s} native_params={result.native_params}")

### Recovery check

Compare the configured mean/variance against the frozen distribution's own `.mean()`/`.var()` (exact) and a large sample's empirical mean/variance (statistical, should be close).

In [ ]:
print(f"{'family':20s} {'mean cfg':>10s} {'mean exact':>11s} {'mean sample':>12s} "
      f"{'var cfg':>10s} {'var exact':>10s} {'var sample':>11s}")
for family, result in results.items():
    cfg = CONFIGS[family]
    sample = result.dist.rvs(size=100_000, random_state=rng)
    print(
        f"{family.value:20s} {cfg['mean']:10.4g} {result.dist.mean():11.4g} {sample.mean():12.4g} "
        f"{cfg['variance']:10.4g} {result.dist.var():10.4g} {sample.var():11.4g}"
    )

### Preview plots

What the distribution-picker widget (PLAN.md section 7) will show live as a user drags mean/variance sliders — a pdf curve for continuous families, a pmf stem plot for discrete ones.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (family, result) in zip(axes.flat, results.items()):
    x, y = preview_xy(result)
    if family.value in ("poisson", "negative_binomial"):
        ax.stem(x, y, basefmt=" ")
    else:
        ax.plot(x, y)
    ax.set_title(f"{family.value}  (mean={CONFIGS[family]['mean']:g}, var={CONFIGS[family]['variance']:g})")
fig.tight_layout()
plt.show()

### Domain validation

Every family has a domain the (mean, variance) pair must fall inside — a Beta mean has to be in
`(0, 1)`, Negative Binomial needs variance strictly greater than mean, etc. These should raise a
clear `DistributionConfigError`, not silently clamp or produce garbage params (PLAN.md section 4).

In [ ]:
from funnel_sim.simulation.distributions import DistributionConfigError

bad_configs = [
    (Family.BETA, 0.5, 0.3, "Beta variance too large for the mean"),
    (Family.BETA, 1.5, 0.01, "Beta mean outside (0, 1)"),
    (Family.NEGATIVE_BINOMIAL, 5.0, 4.0, "Neg-Binomial variance <= mean"),
    (Family.GAMMA, 3.0, -1.0, "negative variance"),
]
for family, mean, variance, label in bad_configs:
    try:
        moments_to_params(family, mean, variance)
        print(f"{label}: NO ERROR RAISED (bug!)")
    except DistributionConfigError as e:
        print(f"{label}: raised cleanly -> {e}")

In [ ]:
import warnings

# Poisson's one free parameter: mean == variance always. Passing a different variance should
# warn (not error) and fall back to variance == mean.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = moments_to_params(Family.POISSON, mean=5.0, variance=20.0)
print(f"Poisson dist mean={result.dist.mean():.3g}, var={result.dist.var():.3g}")
print(f"Warnings raised: {[str(w.message) for w in caught]}")

## 2. Effects system — logit-space composition

A base close-probability plus three stacked effects, applied additively in logit space and
converted back to a probability with sigmoid — see `funnel_sim.simulation.effects`.

In [ ]:
base_rate = 0.20  # base close probability for an "average" opportunity

effects = (
    Effect(name="rep_skill_above_average", logit_delta=0.40, description="Rep is +1 SD skill"),
    Effect(name="industry_construction", logit_delta=-0.30, description="Historically slower to close"),
    Effect(name="fast_first_touch", logit_delta=0.60, description="Contacted within 5 minutes"),
)
effect_set = EffectSet(base_prob=base_rate, effects=effects)

print(f"base rate:            {base_rate:.1%}")
for e in effects:
    print(f"  + {e.name:28s} logit_delta={e.logit_delta:+.2f}")
print(f"total logit shift:     {effect_set.total_delta():+.3f}")
print(f"composed probability:  {effect_set.apply():.1%}")

### Stress test: does stacking many/large effects stay well-behaved?

In raw probability space this is exactly where naive addition breaks (goes negative or above 100%). In logit space it shouldn't — it should just saturate toward 0 or 1.

In [ ]:
extreme_deltas = [3.0, 2.5, -4.0, 5.0, -1.0, 4.0]  # would sum to +9.5 in "raw %" terms — nonsense there
result = apply_effects(base_rate, extreme_deltas)
print(f"sum of raw deltas (nonsense if added directly): {sum(extreme_deltas):+.2f}")
print(f"composed probability (logit space): {result:.6f}")
assert 0.0 < result < 1.0
assert np.isfinite(result)
print("stays inside (0, 1) and finite -- as expected.")

## 3. The ground-truth recovery demo

This is the core value proposition from PLAN.md section 1: *"you control the generative process,
so you know the ground truth... every analysis in the app can be scored against a known answer."*

Simulate a population of synthetic opportunities with **known, configured** logit-space effects,
sample binary win/loss outcomes from the resulting probabilities, then fit a logistic regression
and check whether it recovers the effects we configured.

In [ ]:
n = 20_000

# Each opportunity independently gets each effect "on" or "off" (a real feature would look this
# up per-row from rep_id/industry/etc.; this is the minimal version that proves composability).
true_effects = {
    "rep_skill_above_average": 0.45,
    "industry_construction": -0.35,
    "fast_first_touch": 0.65,
}
base_rate = 0.18

rep_skill = rng.integers(0, 2, size=n).astype(float)
industry_construction = rng.integers(0, 2, size=n).astype(float)
fast_first_touch = rng.integers(0, 2, size=n).astype(float)

deltas = [
    rep_skill * true_effects["rep_skill_above_average"],
    industry_construction * true_effects["industry_construction"],
    fast_first_touch * true_effects["fast_first_touch"],
]
win_prob = apply_effects(base_rate, deltas)
won = rng.binomial(1, win_prob)

print(f"configured base rate:     {base_rate:.1%}")
print(f"simulated overall win rate: {won.mean():.1%}")

In [ ]:
X = sm.add_constant(np.column_stack([rep_skill, industry_construction, fast_first_touch]))
model = sm.Logit(won, X).fit(disp=0)

feature_names = ["const", "rep_skill_above_average", "industry_construction", "fast_first_touch"]
print(model.summary(xname=feature_names))

In [ ]:
from scipy.special import logit as _logit

print(f"{'effect':28s} {'true (logit)':>13s} {'estimated':>11s} {'p-value':>9s}")
print(f"{'base rate (const)':28s} {_logit(base_rate):13.3f} {model.params[0]:11.3f} {model.pvalues[0]:9.3g}")
for i, name in enumerate(true_effects, start=1):
    print(f"{name:28s} {true_effects[name]:13.3f} {model.params[i]:11.3f} {model.pvalues[i]:9.3g}")

If the estimates above land close to the "true (logit)" column and every effect is significant
(small p-value), that's the ground-truth-recovery story PLAN.md section 1 is built around: *"I
configured rep A to close 15% more often; the fitted model estimated +14.2% (p<0.001)."* This
confirms the two modules compose correctly and are ready to be built on for Phase 1 — the actual
Streamlit UI (distribution picker, effect configuration) can now sit on top of this validated
core rather than being designed alongside it.